# Diabetes Risk Prediction using XGBoost and SHAP
### Single-condition ML prototype — Pima Indians Diabetes dataset

This notebook trains **one diabetes classifier**. It does not train heart, liver or kidney models. SHAP explains one sample prediction; ReportLab creates a diabetes assessment PDF.


## 1. Environment Setup and Data Ingestion
Install the required packages and load the Pima Indians Diabetes dataset from the public GitHub CSV.


In [ ]:
!pip -q install shap reportlab xgboost pandas scikit-learn joblib

import json, joblib
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

data_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigree','Age','Outcome']
df = pd.read_csv(data_url, names=columns)
print('Dataset shape:', df.shape)
display(df.head())


## 2. Model Training
We separate features from the target variable, split the dataset into training and testing sets, and train a powerful gradient-boosted tree model (`XGBClassifier`).

In [ ]:
X = df.drop(columns=['Outcome'])
y = df['Outcome'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]
metrics = {'accuracy':float(accuracy_score(y_test,preds)), 'precision':float(precision_score(y_test,preds,zero_division=0)), 'recall':float(recall_score(y_test,preds,zero_division=0)), 'f1':float(f1_score(y_test,preds,zero_division=0)), 'roc_auc':float(roc_auc_score(y_test,probs)), 'confusion_matrix_tn_fp_fn_tp':confusion_matrix(y_test,preds).ravel().astype(int).tolist(), 'train_rows':int(len(X_train)), 'test_rows':int(len(X_test)), 'random_state':42}
print('Held-out test metrics:')
print(json.dumps(metrics,indent=2))
print('\nClassification report:\n',classification_report(y_test,preds,zero_division=0))
with open('diabetes_evaluation_metrics.json','w') as f: json.dump(metrics,f,indent=2)


## 3. Extracting SHAP Explanations
SHAP (SHapley Additive exPlanations) will break down exactly *why* a specific user input got a certain risk score. This builds absolute transparency and clinical trust.

In [ ]:
# Explain one example using SHAP values from the trained diabetes classifier.
explainer = shap.TreeExplainer(model)
mock_user_input = pd.DataFrame([[2,145,80,25,120,32.5,0.5,45]], columns=X.columns)
risk_probability = float(model.predict_proba(mock_user_input)[0,1] * 100)
raw_shap = explainer.shap_values(mock_user_input)
if isinstance(raw_shap, list):
    shap_row = np.asarray(raw_shap[-1])[0]
elif np.asarray(raw_shap).ndim == 3:
    shap_row = np.asarray(raw_shap)[0,:,1]
else:
    shap_row = np.asarray(raw_shap)[0]
base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = np.asarray(base_value).reshape(-1)[-1].item()
print(f'Predicted diabetes probability for demonstration input: {risk_probability:.2f}%')
shap.initjs()
shap.force_plot(base_value, shap_row, mock_user_input.iloc[0], matplotlib=True)


## 4. Automated PDF Report Generation
We dynamically build a clean, patient-facing PDF report containing risk matrixes, custom dynamic recommendations, and mandatory legal disclaimers using `reportlab`.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

def generate_diabetes_report(filename, risk_score, user_data, shap_contributions, evaluation_metrics):
    doc = SimpleDocTemplate(filename, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40, title='Diabetes Risk Prediction Report')
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('TitleCustom', parent=styles['Heading1'], fontSize=19, textColor=colors.HexColor('#1E3A8A'), spaceAfter=14)
    heading = ParagraphStyle('SectionCustom', parent=styles['Heading2'], fontSize=13, textColor=colors.HexColor('#0D9488'), spaceBefore=12, spaceAfter=7)
    body = ParagraphStyle('BodyCustom', parent=styles['Normal'], fontSize=9, leading=13)
    story = [Paragraph('DIABETES RISK PREDICTION REPORT', title_style), Paragraph('Educational ML output from an XGBoost classifier trained on the Pima Indians Diabetes dataset. This is not a medical diagnosis.', body), Spacer(1,12)]
    story += [Paragraph('1. Demonstration Prediction', heading), Paragraph(f'Predicted diabetes probability for the demonstration input: <b>{risk_score:.2f}%</b>', body), Spacer(1,8)]
    rows = [[Paragraph('<b>Input feature</b>',body),Paragraph('<b>Value</b>',body)]] + [[str(k),str(v)] for k,v in user_data.items()]
    t = Table(rows, colWidths=[240,240]); t.setStyle(TableStyle([('GRID',(0,0),(-1,-1),0.4,colors.grey),('BACKGROUND',(0,0),(-1,0),colors.HexColor('#E5E7EB')),('PADDING',(0,0),(-1,-1),5)])); story += [Paragraph('2. Demonstration Input',heading),t]
    metric_rows = [[Paragraph('<b>Held-out test metric</b>',body),Paragraph('<b>Value</b>',body)]] + [[k,str(round(v,4) if isinstance(v,float) else v)] for k,v in evaluation_metrics.items() if k in ['accuracy','precision','recall','f1','roc_auc','train_rows','test_rows']]
    mt = Table(metric_rows,colWidths=[240,240]); mt.setStyle(TableStyle([('GRID',(0,0),(-1,-1),0.4,colors.grey),('BACKGROUND',(0,0),(-1,0),colors.HexColor('#E5E7EB')),('PADDING',(0,0),(-1,-1),5)])); story += [Paragraph('3. Model Evaluation',heading),mt]
    story += [Paragraph('4. Largest SHAP Contributions',heading), Paragraph('<br/>'.join([f'{k}: {float(v):+.4f}' for k,v in sorted(shap_contributions.items(),key=lambda x:abs(float(x[1])),reverse=True)[:8]]),body), Spacer(1,14), Paragraph('Disclaimer: This educational model is not validated for clinical use. Consult a qualified healthcare professional for diagnosis or treatment.',body)]
    doc.build(story)
    print(f'PDF report exported: {filename}')

user_metrics = mock_user_input.iloc[0].to_dict()
shap_contribs = dict(zip(X.columns, shap_row))
generate_diabetes_report('diabetes_risk_report.pdf', risk_probability, user_metrics, shap_contribs, metrics)
joblib.dump({'model':model,'feature_columns':X.columns.tolist(),'metrics':metrics}, 'diabetes_xgb_model.joblib')
print('Saved diabetes_xgb_model.joblib and diabetes_evaluation_metrics.json')
